# 1. Imports

In [323]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

# 2. Funções

## 2.1. Função para pegar os eventos de uma temporada nos arquivos parquet

In [324]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

# 3. Preparação dos dados

## 3.1. Criação da Sessão Spark

In [325]:
# Criação da sessão Spark local
spark = SparkSession.builder.master("local[*]").appName("season_database").getOrCreate()

## 3.2. Criação do df para pegar os eventos de todas as partidas da temporada de 2022-2023 da Premier League

(dps pode ser interessante levar a parte do schema dos jogadores e da bola p etapa de extração)

In [326]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

# Schema em Pyspark para poder parsear o json dos dados de tracking dos jogadores que está como string
players_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("player", StructType([
            StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        StructField("visibility", StringType(), True),
        StructField("confidence", StringType(), True),
        StructField("jerseyNum", StringType(), True)      
    ])
)

# Schema em Pyspark para poder parsear o json dos dados de tracking da bola que está como string
balls_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("visibility", StringType(), True)
    ])
)

details_schema = MapType(StringType(), StringType())

df_events = df_events.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": from_json("homePlayers", players_schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": from_json("awayPlayers", players_schema),

    # Cria coluna com json parseado para dicionário para dados de tracking da bola
    "balls_parsed": from_json("balls", balls_schema),

    "details_parsed": from_json("details", details_schema)

}).drop('homePlayers', 'awayPlayers', 'balls', 'details')

# df_events.show(5)

In [327]:
print('Quantidade de linhas:', df_events.count())

Quantidade de linhas: 945154


## 3.2. Obter jogos da temporada e ajustar identificação do mandante/adversário

(dps pode ser interessante levar isso p etapa de extração)

In [328]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games = spark.read.csv(games_path, header=True)

df_games_raw = df_games.withColumnRenamed("id","gameId").filter(F.col('season') == '2022-2023')

# se venueType == TEAM_HOME, (homeTeam.id == team.id e homeTeam.name == team.name) e (opponentTeam.id == opponentTeam.id e opponentTeam.name == opponentTeam.name)
# se venueType == OPPONENT_HOME, (homeTeam.id == opponentTeam.id e homeTeam.name == opponentTeam.name) e (opponentTeam.id == team.id e opponentTeam.name == team.name)
df_games = (
    df_games_raw.withColumns({
    "homeTeam.id": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`team.id`')).otherwise(F.col('`opponentTeam.id`')),
    "homeTeam.name": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`team.name`')).otherwise(F.col('`opponentTeam.name`')),

    "opponentTeam.id": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`opponentTeam.id`')).otherwise(F.col('`opponentTeam.id`')),
    "opponentTeam.name": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`opponentTeam.name`')).otherwise(F.col('`opponentTeam.name`')),

}).drop('venueType', 'team.id', 'team.name')
)

df_games.show(5)

+------+----------+---------+----------------------+-------------+--------------+----------------+---------------+--------------------+-------------+--------------+-------------+-----------+--------------------+
|gameId|      date|   season|teamExtraTimeStartSide|teamStartSide|competition.id|competition.name|opponentTeam.id|   opponentTeam.name| stadium.name|stadium.length|stadium.width|homeTeam.id|       homeTeam.name|
+------+----------+---------+----------------------+-------------+--------------+----------------+---------------+--------------------+-------------+--------------+-------------+-----------+--------------------+
|  4447|2022-08-13|2022-2023|                 Right|         Left|             1|  Premier League|              8|             Everton|   Villa Park|         105.0|         68.0|          3|         Aston Villa|
|  4760|2023-04-25|2022-2023|                  Left|         Left|             1|  Premier League|             20|Wolverhampton Wan...|     Molineux|   

In [329]:
#df_events_games = df_events.join(df_games, on = "gameId", how='left')
#df_events_games.show()

## Domínios

In [330]:
# tabelas:
# partida - match_id
# competition -> competition_id
# season -> season_id
# events -> event_id

# df_defensive_events (domínio de desempenho técnico defensivo):
# competition_id | season_id | match_id | event_id | period | match_timestamp | event_type | event_name | atk_team | def_team | match_time_atk | match_time_def

# cruzamento dos dfs (pelo competitionID-seasonID-matchID) -> apenas deixar preparado para qnd fosse expandir, mas vamos usar o mesmo competitionID-seasonID:

# cria df_defensive_events -> cria df_threat_events -> outer join ou union dos dois pelo competitionID-seasonID-matchID (apenas deixar preparado para qnd fosse expandir, mas vamos usar o mesmo competitionID-seasonID) -> calcular o diff_threat_score

### Domínio: Desempenho Técnico Defensivo

In [331]:
df_events.show(5)

+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------+----------------+------------+--------------+--------------------+--------------------+--------------------+--------------------+
|             eventId|competitionId|gameId|   season|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|eventPlayer.id|eventPlayer.name|eventTeam.id|eventTeam.name|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      details_parsed|
+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------+----------------+------------+--------------+--------------------+--------------------+--------------------+--------------------+
|9e6a498f54bad910e...|            1|  4438|2022-2023|     1|       First half|       

In [332]:
df_events = df_events.select(
    'competitionId',
    'season', # dps mudar pra seasonId se necessário
    'gameId',
    'eventId',
    'eventType',
    'eventTypeDescription',
    'period',
    #'periodDescription',
    #'startFormattedGameClock',
    'startGameClock',
    'details_parsed',
    F.col('`eventPlayer.id`').alias('eventPlayerId'),
    F.col('`eventPlayer.name`').alias('eventPlayerName'),
    F.col('`eventTeam.id`').alias('eventTeamId'), 
    F.col('`eventTeam.name`').alias('eventTeamName'), 
    #'homePlayers_parsed', 
    #'awayPlayers_parsed', 
    #'balls_parsed'
) #.filter(F.col('gameId') == 4438) # filtro pra só um jogo por enquanto

### Tipos de eventos:

- FIRSTKICKOFF: Inicio do primeiro tempo
- SECONDKICKOFF: Inicio do segundo tempo
- TC: Touch
- RE: Rebound
- BC: Ball Carry
- CL: Clearance
- CR: Cross
- CH: Challenge 
- OTB: A possession with a player on the ball
- PA: Pass
- FO: Foul
- FOUL: Additional foul
- SH: Shot

### Eventos que queremos (Defensivos):

- Ofensivos como CR, PA e SH queremos que o resultado dele seja uma interferência da defesa adversária
- Defensivos como CL, CH, FO queremos que o tipo seja ação defensiva

- CL: Clearance
    - Qualquer CLEARANCE_OUTCOME_TYPE (A,B,D,E,O,P,S,U)
    - obs: talvez não E e U pq são FairPlay
    - obs2: P - Player e S - Stoppage não sei oq sejam, mas vou deixar

- CR: Cross
    - CROSS_OUTCOME_TYPE:
    - B - Blocked
    - D - Defensive Interception

- CH: Challenge. 
    - CHALLENGE_TYPE:
    - ‘5’ - 50/50. This is a duel type where two players compete for a loose ball.
    - A - Aerial duel. As the name suggests a duel type similar to 50-50, but with the ball coming from above.
    - B - Tackle from behind. As the name suggests a tackle attempt where the carrier puts their body in between the ball and the challenger as the tackle is attempted.
    - D - Dribble. The player tries to take on a defender in an attempt to get past them.
    - G - Goalkeeper smothers ball. A duel between the goalkeeper and a line player where the ball is loose and the goalkeeper tries to capture the ball.
    - H - Shielding. Similar to tackle from behind, but on a shielding challenge the carrier actively shields a defender who does not attempt a tackle
    - K - Hand tackle by goalkeeper. Despite the name, it is a duel type similar to goalkeeper smothers, but the keeper tries to parry the ball rather than retain it.
    - L - Slide tackle. Tackle type where the challenger slides to attempt to win the ball. Note that a player could be sliding on a dribble or 50-50, to be classed as a slide tackle it needs to be first and foremost a tackle.
    - S - Shoulder to shoulder. Tackle type where the challenger tries to win the ball with physical contact initiated with the body.
    - T - Standing tackle. Tackle attempt, usually from the front or side, that does not fit the other tackle types 
    - OBS1: **Único que não entraria como AD aqui seria o 'D'.**
    - OBS2: **Não estamos considerando outcome dos eventos.**

- PA: Pass
    - PASS_OUTCOME_TYPE:
    - B - Blocked
    - D - Defensive Interception

- FO: Foul
    - qualquer FOUL_TYPE = A, I, M
    - não vi evento de penalti, então teria q pegar a região dentro da area e evento de falta marcado ali (FOUL_TYPE == I)

- FOUL: Additional foul
    - são faltas adicionais no mesmo lance divida em mais de um evento, mas nos dados fica tudo NULL, então n vou add. FO já tem o evento principal de falta

- SH: Shot
    - SHOT_OUTCOME_TYPE:
    - B - Block on target. (Ball was going on target, but got blocked)
    - C - Block off target. (Ball was going off target, and got blocked)
    - F - Save off target. (Ball was going off target when it got saved)
    - L - Goalline clearance. (Ball is past the goalkeeper and a defender stops it from going into the net)
    - S - Save on target. (Ball was going on target and got saved).

In [333]:
desired_events = ['CL', 'CR', 'CH', 'PA', 'FO', 'SH'] # não inclui FO pq tá estranho misturando eventos dos outros e não de falta em si (dps validar isso)

cl_cond = (
    (F.col("eventType") == "CL") &
    F.col("details_parsed")["clearanceOutcomeType"].isin(['A','B','D','O','P','S'])
)

cr_cond = (
    (F.col("eventType") == "CR") &
    F.col("details_parsed")["crossOutcomeType"].isin(['B','D'])
)

ch_cond = (
    (F.col("eventType") == "CH") &
    ~F.col("details_parsed")["challengeType"].isin(['D'])
)

pa_cond = (
    (F.col("eventType") == "PA") &
    F.col("details_parsed")["passOutcomeType"].isin(['B','D'])
)

sh_cond = (
    (F.col("eventType") == "SH") &
    F.col("details_parsed")["shotOutcomeType"].isin(['B','C','F','L','S'])
)

In [334]:
df_defensive_events = (
    df_events
    .filter(F.col("eventType").isin(desired_events))
    .withColumns({
        "defensiveType": (
            F.when(cl_cond, F.col("details_parsed")["clearanceOutcomeType"])
            .when(cr_cond, F.col("details_parsed")["crossOutcomeType"])
            .when(ch_cond, F.col("details_parsed")["challengeType"])
            .when(pa_cond, F.col("details_parsed")["passOutcomeType"])
            .when(sh_cond, F.col("details_parsed")["shotOutcomeType"])
        ),

        "defensiveDescription": (
            F.when(cl_cond, F.col("details_parsed")["clearanceOutcomeTypeDescription"])
            .when(cr_cond, F.col("details_parsed")["crossOutcomeTypeDescription"])
            .when(ch_cond, F.col("details_parsed")["challengeTypeDescription"])
            .when(pa_cond, F.col("details_parsed")["passOutcomeTypeDescription"])
            .when(sh_cond, F.col("details_parsed")["shotOutcomeTypeDescription"])
        )
    }).dropna(subset=['defensiveType'])
)

df_defensive_events.show()

+-------------+---------+------+--------------------+---------+--------------------+------+--------------+--------------------+-------------+-----------------+-----------+---------------+-------------+--------------------+
|competitionId|   season|gameId|             eventId|eventType|eventTypeDescription|period|startGameClock|      details_parsed|eventPlayerId|  eventPlayerName|eventTeamId|  eventTeamName|defensiveType|defensiveDescription|
+-------------+---------+------+--------------------+---------+--------------------+------+--------------+--------------------+-------------+-----------------+-----------+---------------+-------------+--------------------+
|            1|2022-2023|  4438|a084905de5dc02ce4...|       PA|                Pass|     1|             4|{blockerPlayer ->...|         1896|     Matthew Cash|          3|    Aston Villa|            D|Defensive Interce...|
|            1|2022-2023|  4438|751b1b462a689cb87...|       CH|           Challenge|     1|            29|{i

In [ ]:
# df_defensive_events (domínio de desempenho técnico defensivo):
# competition_id | season_id | match_id | event_id | period | match_timestamp | event_type | event_name | atk_team | def_team | match_time_atk | match_time_def

### Domínio: Ameaça

In [38]:
# df_threat_events (dominio de ameaça):
# competition_id | season_id | match_id | event_id | period | match_timestamp | event_type | event_name | atk_team | def_team | atk_tracking | def_tracking | abs_total_players_atk | abs_total_players_def | num_diff (atk | def) | dist_gol | threat_score 